# detach-stop-gradient-trick — worked example 3: Use torch.no_grad() as the cheaper D-step stop-gradient

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `detach-stop-gradient-trick`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`torch.no_grad()` is the second classic way to keep the generator out of the discriminator's update. Inside a `with t.no_grad():` block, PyTorch builds no autograd graph at all, so `G(z)` comes out with `requires_grad=False`. This is cheaper than `.detach()` because the graph for G's forward is never constructed in the first place, yet the discriminator loss and its gradients are identical.

## Worked solution

**Goal:** generate fakes under `no_grad`, run D's loss, and confirm G stays gradient-free while the loss matches the detach approach.

1. **Build G and D** as small linears after seeding so the run is deterministic.
2. **Generate inside `no_grad`.** `with t.no_grad(): fake = G(z)`. Within this context manager autograd is globally suppressed, so `fake.requires_grad` is `False` and no graph node is recorded for G's matmul.
3. **Compute D's loss outside the block.** `loss = (D(fake) - D(x_real)).mean()`. Now autograd is active again, so D's contribution IS recorded — only G's part was skipped.
4. **Backward.** `loss.backward()` flows into D's parameters only; G's parameters keep `.grad is None` because there was never a path to them.
5. **Cross-check against detach.** Re-running the same forward with `G(z).detach()` yields a loss within floating-point tolerance, confirming the two idioms are equivalent for the D-step.

We print the no_grad loss, the detach loss, their closeness, and whether any G grad was touched.

In [ ]:
import torch.nn as nn

t.manual_seed(0)
G = nn.Linear(4, 4)
D = nn.Linear(4, 1)
z = t.randn(8, 4)
x_real = t.randn(8, 4)

for p in list(G.parameters()) + list(D.parameters()):
    p.grad = None
with t.no_grad():
    fake = G(z)
loss_ng = (D(fake) - D(x_real)).mean()
loss_ng.backward()
g_touched = any(p.grad is not None for p in G.parameters())

for p in list(G.parameters()) + list(D.parameters()):
    p.grad = None
fake2 = G(z).detach()
loss_dt = (D(fake2) - D(x_real)).mean()
loss_dt.backward()

print('no_grad loss:', round(loss_ng.item(), 5))
print('detach  loss:', round(loss_dt.item(), 5))
print('losses close:', abs(loss_ng.item() - loss_dt.item()) < 1e-5)
print('fake.requires_grad:', fake.requires_grad)
print('any G grad non-None:', g_touched)